# CMSC 173 &middot; Machine Learning &mdash; Week 3 Lab
## Linear Regression: Fitting a Line by Gradient Descent

Last week you *estimated* parameters. This week you **fit a model**: a straight line through
data. You'll build the two engines that do it &mdash; **gradient descent** (take small downhill
steps) and the **normal equation** (solve it in one shot) &mdash; entirely from scratch, and watch
gradient descent converge on a graph.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib only (no scikit-learn until week 4).** **Not graded.** About 55 minutes.

---
## Part 0 &middot; Setup

Run once.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(173)
print('Ready.')

---
## Part 1 &middot; The data and the model

We invent data with a rule we *know*, so we can check whether the model recovers it. Say a
student's exam score rises with hours studied per day:

$$\text{score} = \theta_0 + \theta_1 \cdot \text{hours} + \text{noise}.$$

$\theta_0$ (intercept) is the score with zero study; $\theta_1$ (slope) is how many points each
extra hour buys. Fitting = finding the $\theta_0, \theta_1$ that best match the dots.

In [ ]:
true_b, true_w = 40.0, 8.0                       # (1) the TRUE intercept and slope
hours = rng.uniform(0, 5, size=60)              # (2) 60 students, 0-5 hours/day
score = true_b + true_w*hours + rng.normal(0, 6, size=60)   # (3) rule + random noise

plt.figure(figsize=(7,4))                        # (4) look at the data
plt.scatter(hours, score, alpha=0.7)
plt.xlabel('hours studied / day'); plt.ylabel('exam score')
plt.title('The data we will fit a line to'); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** the true intercept (40) and slope (8) &mdash; the answer we hope to recover.
- **(2)** `rng.uniform(0,5,60)` makes 60 study-hour values between 0 and 5.
- **(3)** apply the rule and add `rng.normal(0,6,...)` noise so the dots scatter around the line,
  like real data.
- **(4)** a scatter plot: an upward cloud. Our job is to draw the best line through it.

**Answer here:**

1. Just by eye, roughly what score would you predict for 3 hours? Is the relationship positive?
   &rarr; *your answer*

2. Why add noise in step (3) instead of using a perfect line? What would over-trusting a perfect
   fit be called (you saw the word last week)?
   &rarr; *your answer*

---
## Part 2 &middot; How wrong is a line? The cost function

A line makes a prediction for each student; the **cost** measures how far off those predictions
are, all together. We use **Mean Squared Error**: average the squared gaps. Squaring punishes
big misses and keeps signs from cancelling. Lower cost = better line.

In [ ]:
def predict(hours, b, w):        # (1) the line's prediction
    return b + w*hours

def cost(hours, score, b, w):    # (2) mean squared error
    errors = predict(hours, b, w) - score
    return np.mean(errors**2)

print('cost of the TRUE line :', round(cost(hours, score, true_b, true_w), 2))
print('cost of a BAD line    :', round(cost(hours, score, 10, 2), 2))   # (3) worse line, higher cost

**Reading the code, line by line:**
- **(1)** `predict` is literally $b + w\cdot\text{hours}$, computed for every student at once.
- **(2)** `cost` subtracts the true scores from the predictions, squares each gap, and averages.
- **(3)** the true line scores a *low* cost; a badly chosen line (b=10, w=2) scores much higher.
  Fitting = searching for the line with the smallest cost.

**Answer here:**

1. Change the BAD line's numbers to get its cost *below* the true line's, if you can. Can you beat
   it by much? What does that tell you about where the minimum is?
   &rarr; *your answer*

---
## Part 3 &middot; Gradient descent (the core idea)

We won't guess lines by hand. **Gradient descent** starts anywhere and repeatedly steps *downhill*
on the cost. The gradient is the slope of the cost surface; we nudge each parameter against it.
For MSE the two gradients are:

$$\frac{\partial\,\text{cost}}{\partial b} = \frac{2}{m}\sum(\hat y - y), \qquad
\frac{\partial\,\text{cost}}{\partial w} = \frac{2}{m}\sum(\hat y - y)\,x.$$

The **learning rate** controls step size. We'll record the cost every step and graph it.

In [ ]:
def gradient_descent(hours, score, lr=0.03, n_iters=800):
    b, w = 0.0, 0.0                       # (1) start at a flat line
    m = len(score)
    history = []                          # (2) cost after each step
    for _ in range(n_iters):
        err  = predict(hours, b, w) - score      # (3) prediction minus truth
        grad_b = (2/m) * err.sum()               # (4) gradient wrt intercept
        grad_w = (2/m) * (err * hours).sum()     # (5) gradient wrt slope
        b -= lr * grad_b                          # (6) step downhill
        w -= lr * grad_w
        history.append(cost(hours, score, b, w)) # (7) remember the cost
    return b, w, np.array(history)

b_hat, w_hat, hist = gradient_descent(hours, score)
print(f'gradient descent found: intercept={b_hat:.2f}, slope={w_hat:.2f}')
print(f'the truth was:          intercept={true_b}, slope={true_w}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))    # (8) two graphs side by side
ax[0].plot(hist); ax[0].set_xlabel('iteration'); ax[0].set_ylabel('cost (MSE)')
ax[0].set_title('Cost falls as we step downhill')
ax[1].scatter(hours, score, alpha=0.6)
xs = np.array([0, 5]); ax[1].plot(xs, predict(xs, b_hat, w_hat), 'r-', lw=2, label='fitted line')
ax[1].set_xlabel('hours'); ax[1].set_ylabel('score'); ax[1].set_title('The fitted line'); ax[1].legend()
plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** start from a flat line (both parameters zero).
- **(2)** `history` will hold the cost after every step so we can graph the descent.
- **(3)** `err` = how far each prediction is above/below the true score.
- **(4)&ndash;(5)** the two gradient formulas from the explainer, in code: sum the errors (for the
  intercept) and the errors weighted by `hours` (for the slope).
- **(6)** `b -= lr*grad_b` takes a small step *against* the gradient &mdash; downhill.
- **(7)** record the new cost.
- **(8)** left graph: the cost curve drops and flattens (that flattening = convergence). Right
  graph: the recovered line sits right through the cloud, close to the true 40 + 8&middot;hours.

**Answer here:**

1. Change `lr=0.03` to `lr=0.3` and run. What happens to the cost curve? Now try `lr=0.001`. In
   one sentence: what does the learning rate trade off? (Then put it back to 0.03.)
   &rarr; *your answer*

2. The cost curve flattens but never hits exactly zero. Why not &mdash; what in the data stops a
   straight line from being perfect?
   &rarr; *your answer*

---
## Part 4 &middot; Why we scale features

With one small feature, gradient descent was easy. Real features live on wildly different scales
(age in years vs income in pesos), and that makes the cost surface a stretched valley gradient
descent zig-zags down slowly. The fix is **standardising**: subtract the mean, divide by the
standard deviation, so every feature has a comparable size. Watch the two cost curves.

In [ ]:
big = hours * 1000                                   # (1) same feature, but on a huge scale
_, _, hist_raw = gradient_descent(big, score, lr=1e-7, n_iters=800)   # (2) needs a tiny lr

z = (big - big.mean()) / big.std()                   # (3) standardise: mean 0, std 1
_, _, hist_scaled = gradient_descent(z, score, lr=0.3, n_iters=800)   # (4) can use a big lr

plt.figure(figsize=(7,4))
plt.plot(hist_raw,    label='un-scaled feature (crawls)')
plt.plot(hist_scaled, label='standardised feature (dives)')
plt.xlabel('iteration'); plt.ylabel('cost'); plt.yscale('log')
plt.title('Standardising lets gradient descent converge fast')
plt.legend(); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** we blow up the feature by 1000&times; to mimic a large-scale variable.
- **(2)** on the raw huge feature, gradient descent only stays stable with a *microscopic* learning
  rate, so it barely moves &mdash; the cost crawls.
- **(3)** standardising `(x - mean)/std` recentres and rescales it to a friendly range.
- **(4)** now a normal-sized learning rate works and the cost **dives**. Same data, same model &mdash;
  only the scaling changed. (The y-axis is log so both curves fit on one plot.)

**Answer here:**

1. Both runs fit the *same* underlying relationship. Which converged in fewer iterations, and by
   roughly how much? State the practical rule in one sentence.
   &rarr; *your answer*

---
## Part 5 &middot; The one-shot answer: the normal equation

Gradient descent *searches*. For linear regression there is also an exact formula that jumps
straight to the best parameters:

$$\theta = (X^\top X)^{-1} X^\top y,$$

where $X$ stacks a column of 1s (for the intercept) next to the feature. No learning rate, no
iterations &mdash; but it inverts a matrix, which is slow when you have thousands of features.

In [ ]:
X = np.column_stack([np.ones_like(hours), hours])    # (1) [1, hours] design matrix
theta = np.linalg.solve(X.T @ X, X.T @ score)        # (2) solve, don't invert (faster, safer)

print(f'normal equation : intercept={theta[0]:.2f}, slope={theta[1]:.2f}')
print(f'gradient descent: intercept={b_hat:.2f}, slope={w_hat:.2f}')
print(f'the truth       : intercept={true_b}, slope={true_w}')

**Reading the code, line by line:**
- **(1)** `column_stack([ones, hours])` builds $X$: first column all 1s (so $\theta_0$ is the
  intercept), second column the feature.
- **(2)** `np.linalg.solve(A, b)` solves $A\theta = b$ with $A = X^\top X$ and $b = X^\top y$. It
  gives the same answer as $(X^\top X)^{-1}X^\top y$ but is faster and numerically safer than
  literally inverting.
- The three lines print nearly the same intercept and slope: **two methods, one answer**, both
  close to the truth.

**Answer here:**

1. The normal equation and gradient descent agree here. Name one situation from lecture where you
   would *prefer* gradient descent even though the exact formula exists.
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| Reading a scatter plot | - |
| The idea of a cost function (MSE) | - |
| What gradient descent does, in words | - |
| Why we standardise features | - |
| The normal equation (X^T X)^{-1} X^T y | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: what is gradient descent actually doing each step?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; Two features at once

Real models use many features. Predict score from **both** hours studied and hours slept. The
normal equation code barely changes &mdash; just add a column. Fill in the one missing line.

In [ ]:
sleep = rng.uniform(4, 9, size=60)
score2 = 20 + 8*hours + 3*sleep + rng.normal(0, 5, size=60)   # true: intercept 20, slopes 8 and 3

# your code here: build X2 = [1, hours, sleep] and solve for theta2, then print it
# hint: np.column_stack([np.ones_like(hours), hours, sleep])


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 3

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/3/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 3 submission page](https://portal.latarak.com/course/cmsc173/lab/3/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.